In [5]:
#FULL PIPELINE (URL --> SWE)
import os, tarfile, gzip, shutil, calendar, requests, re, time
import rasterio
import rasterstats
from rasterio.transform import from_origin
from datetime import datetime
from pathlib import Path
from shapely.geometry import mapping, Point
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# --- Directories ---
data_dir = Path('../data/SNODAS') #change to your personal directory
data_dir.mkdir(exist_ok=True, parents=True)

# --- Spatial parameters of SNODAS ---
ul_lat, ul_lon = 52.87500000000000, -124.73333333333333
lr_lat, lr_lon = 24.95000000000000, -66.94166666666667
nrows, ncols = 3351, 6935
res_x = (lr_lon - ul_lon) / ncols
res_y = (ul_lat - lr_lat) / nrows
transform = from_origin(ul_lon, ul_lat, res_x, res_y)
crs = 'EPSG:4326'
nodata = -9999

# --- HDR content (ENVI format) ---
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

# --- Shower Falls point ---
lat = 45.4
lon = -110.95
point_geom = mapping(Point(lon, lat))

# --- URL builders ---
def date_url(date: datetime) -> str:
    url = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/"
    url += f"{date.year}/{date.month:02}_{calendar.month_abbr[date.month]}"
    url += f"/SNODAS_{date.strftime('%Y%m%d')}.tar"
    return url

def urls_for_date_range(start_date: datetime, end_date: datetime) -> list:
    dates = pd.date_range(start_date, end_date, freq='1D')
    return [date_url(date) for date in dates]

# --- Network Retry Loop  ---
class DownloadFailedError(Exception):
    pass

def download_with_retry(url: str, out_path: Path, max_retries: int = 5):
    time.sleep(1) #small delay
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=60)
            with open(out_path, 'wb') as f:
                f.write(response.content)
            return True
        except Exception as e:
            print(f'Attempt {attempt} failed: {e}')
            if out_path.exists():
                os.remove(out_path)
            time.sleep(5 * attempt)
    raise DownloadFailedError(f'Failed after {max_retries} attempts: {url}')

# --- Main pipeline ---
# Thread-safe list for collecting rows
rows_lock = threading.Lock()
rows = []

def process_date(url: str) -> dict:
    filename = url.split('/')[-1]
    tar_path = data_dir / filename

    # 1. Download tar
    print(f'Downloading: {filename}')
    success = download_with_retry(url, tar_path)
    if not success:
        return None

    # 2. Extract 1034 .dat.gz
    with tarfile.open(tar_path, 'r') as tar:
        swe_member = next(
            (m for m in tar.getmembers() if '1034' in m.name and m.name.endswith('.dat.gz')),
            None
        )
        if swe_member is None:
            print(f'No SWE file found in {filename}')
            os.remove(tar_path)
            return None
        tar.extract(swe_member, path=data_dir)

    dat_gz_path = data_dir / swe_member.name

    # 3. Decompress .dat.gz to .dat
    dat_path = dat_gz_path.with_suffix('')
    with gzip.open(dat_gz_path, 'rb') as gz_in:
        with open(dat_path, 'wb') as dat_out:
            shutil.copyfileobj(gz_in, dat_out)

    # 4. Write .hdr
    hdr_path = dat_path.with_suffix('.hdr')
    with open(hdr_path, 'w') as f:
        f.write(hdr_content)

    # 5. Convert .dat to .tif
    tif_path = dat_path.with_suffix('.tif')
    with rasterio.open(dat_path) as src:
        data = src.read()
        width = src.width
        height = src.height
    with rasterio.open(
        tif_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1, dtype=data.dtype,
        crs=crs, transform=transform,
        nodata=nodata
    ) as dst:
        dst.write(data)

    # 6. Query SWE at Shower Falls
    match = re.search(r'TS(\d{8})', dat_path.stem)
    date = pd.to_datetime(match.group(1), format='%Y%m%d')
    result = rasterstats.point_query(
        [point_geom],
        str(tif_path),
        nodata=-9999,
        interpolate='nearest'
    )[0]
    swe = result / 1000 if result is not None else None

    # 7. Delete all intermediates including .tif
    for f in [tar_path, dat_gz_path, dat_path, hdr_path, tif_path]:
        if f.exists():
            os.remove(f)

    print(f'Queried: {dat_path.stem} — SWE: {swe} m')
    return {'date': date, 'swe_m': swe}

# --- Run for date range ---
urls = []
for year in range(2004, 2005):
    year_urls = urls_for_date_range(
        datetime(year, 3, 1),
        datetime(year, 6, 30)
    )
    urls.extend(year_urls)
    
start_time = time.time()
rows = []
failed_url = None

try:
    with ThreadPoolExecutor(max_workers=1) as executor: # I only had success with 1 worker, going one year at a time due to server blocking me.
        futures = {executor.submit(process_date, url): url for url in urls}
        for future in as_completed(futures):
            try:
                result = future.result()
                if result is not None:
                    rows.append(result)
            except DownloadFailedError as e:
                failed_url = futures[future]
                print(f'Download failed, stopping: {failed_url}')
                executor.shutdown(wait=False, cancel_futures=True)
                break
finally:
    # Always save whatever was collected
    if rows:
        df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
        out_csv = data_dir / 'SNODAS_swe_2004.csv'
        if out_csv.exists():
            existing_df = pd.read_csv(out_csv, parse_dates=['date'])
            df = pd.concat([existing_df, df]).sort_values('date').reset_index(drop=True)
        df.to_csv(out_csv, index=False)
        elapsed = time.time() - start_time
        print(f'Saved {len(df)} total records to {out_csv}')
        print(f'Elapsed time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
        if failed_url:
            print(f'Stopped at: {failed_url}')

Downloading: SNODAS_20040301.tar
Attempt 1 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd30a5810>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 2 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd2558550>: Failed to establish a new connection: [Errno 111] Connection refused'))
Attempt 3 failed: HTTPSConnectionPool(host='noaadata.apps.nsidc.org', port=443): Max retries exceeded with url: /NOAA/G02158/masked/2004/03_Mar/SNODAS_20040301.tar (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fdcd23e5b90>: Failed to establish a new connection: [Errno 111]

NameError: name 'df' is not defined

In [2]:
#Combine SNODAS data into 1 .csv
import pandas as pd
import glob
import os

# Load all SNODAS yearly CSVs and combine
snodas_files = glob.glob(r'../data/SNODAS/SNODAS_swe_[0-9]*.csv')
snodas_df = pd.concat([pd.read_csv(f, parse_dates=['date']) for f in snodas_files])
snodas_df = snodas_df.sort_values('date').reset_index(drop=True)
snodas_df = snodas_df.rename(columns={'swe_m': 'swe_m_snodas'})

print(f'{len(snodas_df)} total records')
print(snodas_df.head())

# Save
snodas_df.to_csv(r'../data/SNODAS/SNODAS_swe_all.csv', index=False)

2552 total records
        date  swe_m_snodas
0 2004-03-01         0.325
1 2004-03-02         0.353
2 2004-03-03         0.353
3 2004-03-04         0.353
4 2004-03-05         0.369
